# Phase 2: Feature Engineering and Resampling

This notebook builds the Final Feature Matrix from the Gold Dataset (8,268 seniors) using hrp_processed.db. We'll:

1. **Resample** vitals into 15-minute buckets (mean for HR/BP/Temp/Sat, sum for Steps)
2. **Pivot** from long to wide format
3. **Impute** missing values with forward fill (1 hour limit)
4. **Engineer features**: hr_volatility, bp_trend, pulse_pressure
5. **Fuse static data**: 11 clinical domain flags
6. **Create alert context**: recent_event_burden (Severity 1/2 alerts in past 48h)
7. **Label targets**: label_1 / label_2 / label_3 = 1 if within 24h before Severity 1 / 2 / 3 alerts
8. **Save output**: data/processed/multimodal_features.parquet

Processing senior-by-senior to manage memory for the large final matrix.

## Section 1: Import Libraries

In [62]:
import os
import time
import glob
import sqlite3
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import timedelta
import warnings
import pickle
import json
warnings.filterwarnings('ignore')

sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (14, 8)

## Section 2: Load Gold Dataset and Database Connection

In [25]:
db_path = '../data/processed/hrp_processed.db'
conn = sqlite3.connect(db_path)
cursor = conn.cursor()

print(f"Connected to: {db_path}")

Connected to: ../data/processed/hrp_processed.db


In [32]:
gold_dataset_pickle_path = '../data/processed/gold_seniors.pkl'
gold_dataset_csv_path = '../data/processed/gold_seniors.csv'
metadata_path = '../data/processed/gold_dataset_metadata.json'

with open(gold_dataset_pickle_path, 'rb') as f:
    gold_seniors = pickle.load(f)

gold_seniors_list = list(gold_seniors)

In [38]:
len(gold_seniors)

8268

In [30]:
with open(metadata_path, 'r') as f:
    gold_metadata = json.load(f)

print("Gold Dataset Metadata:")
print(f"  - Total Seniors: {gold_metadata['total_seniors']:,}")
print(f"  - Retention Rate: {gold_metadata['retention_rate_percent']:.1f}%")
print(f"  - Severity 3 Alerts Retained: {gold_metadata['severity_3_alerts']:,} ({gold_metadata['severity_3_retention_percent']:.1f}%)")
print("  - Criteria:")
print(f"    * Criterion 1 ({gold_metadata['criterion_1_count']:,}): {gold_metadata['criteria_description']['criterion_1']}")
print(f"    * Criterion 2 ({gold_metadata['criterion_2_count']:,}): {gold_metadata['criteria_description']['criterion_2']}")

Gold Dataset Metadata:
  - Total Seniors: 8,268
  - Retention Rate: 64.6%
  - Severity 3 Alerts Retained: 47 (77.0%)
  - Criteria:
    * Criterion 1 (8,267): Seniors with >30% overall data density
    * Criterion 2 (26): Seniors with >80% local density in 6 hours before Severity 3 alert


## Section 3: Create Database Index

Before processing 8,268 seniors with 70M+ measurements, we need an index on senior_id to avoid full table scans.

In [41]:
index_check = cursor.execute("SELECT name FROM sqlite_master WHERE type='index' AND tbl_name='measurements'").fetchall()
print(f"Existing indexes: {[idx[0] for idx in index_check]}")

try:
    cursor.execute("CREATE INDEX IF NOT EXISTS idx_measurements_senior_id ON measurements(senior_id)")
    conn.commit()
except Exception as e:
    print(f"Index creation: {e}")

index_verify = cursor.execute("SELECT name FROM sqlite_master WHERE type='index' AND tbl_name='measurements'").fetchall()
print(f"\nCurrent indexes: {[idx[0] for idx in index_verify]}")

Existing indexes: ['idx_measurements_senior', 'idx_measurements_date', 'idx_measurements_senior_date', 'idx_measurements_senior_id']

Current indexes: ['idx_measurements_senior', 'idx_measurements_date', 'idx_measurements_senior_date', 'idx_measurements_senior_id']


## Section 4: Resample Vitals into 15-Minute Buckets

In [47]:
def resample_senior_vitals(senior_id, conn):
    """Resample a single senior's measurements into 15-minute buckets."""
    query = """
    SELECT date, type, value, sbp, dbp, pulse_pressure
    FROM measurements
    WHERE senior_id = ?
    ORDER BY date
    """
    df = pd.read_sql_query(query, conn, params=(str(senior_id),))
    if len(df) == 0:
        return None

    df['date'] = pd.to_datetime(df['date'])
    agg_funcs = {'Heartrate': 'mean', 'Temperature': 'mean', 'Saturation': 'mean', 'Steps': 'sum'}
    df.rename(columns={'type': 'measurement_type'}, inplace=True)
    df.set_index('date', inplace=True)

    resampled_dfs = []
    for mtype in df['measurement_type'].unique():
        df_type = df[df['measurement_type'] == mtype].copy()
        if mtype == 'BloodPressure':
            resampled_dfs.append(df_type['sbp'].resample('15min').mean().to_frame(name='sbp'))
            resampled_dfs.append(df_type['dbp'].resample('15min').mean().to_frame(name='dbp'))
            if 'pulse_pressure' in df_type.columns:
                resampled_dfs.append(df_type['pulse_pressure'].resample('15min').mean().to_frame(name='pulse_pressure'))
        else:
            agg_func = agg_funcs.get(mtype, 'mean')
            resampled_dfs.append(df_type['value'].resample('15min').agg(agg_func).to_frame(name=mtype.lower()))

    if len(resampled_dfs) == 0:
        return None

    result = pd.concat(resampled_dfs, axis=1).reset_index()
    result['senior_id'] = senior_id
    return result

## Section 5: Pivot Data from Long to Wide Format

In [48]:
def pivot_to_wide_format(df_resampled):
    """Transform resampled data from long to wide format."""
    if df_resampled is None or len(df_resampled) == 0:
        return None
    
    df_wide = df_resampled.copy()
    df_wide.rename(columns={'date': 'timestamp'}, inplace=True)
    
    df_wide.columns = [col.lower().replace(' ', '_') for col in df_wide.columns]
    
    return df_wide

## Section 6: Apply Forward Fill Imputation

In [49]:
def apply_forward_fill_imputation(df_wide, limit_buckets=4):
    """Apply forward fill imputation with a limit of 4 buckets (1 hour at 15-min intervals)."""
    if df_wide is None or len(df_wide) == 0:
        return df_wide
    
    df_filled = df_wide.copy()
    
    vital_cols = [col for col in df_filled.columns 
                  if col not in ['timestamp', 'senior_id']]
    
    for col in vital_cols:
        df_filled[col] = df_filled[col].fillna(method='ffill', limit=limit_buckets)
    
    return df_filled

## Section 7: Engineer Signal Features (HR Volatility & BP Trend)

In [53]:
def engineer_signal_features(df_filled):
    """
    Add engineered signal features:
    - hr_volatility: 4-hour rolling standard deviation of HR (16 buckets at 15-min)
    - bp_trend: Slope of SBP over last 3 hours (12 buckets at 15-min)
    """
    
    if df_filled is None or len(df_filled) == 0:
        return df_filled
    
    df_features = df_filled.copy()
    
    if 'heartrate' in df_features.columns:
        df_features['hr_volatility'] = df_features['heartrate'].rolling(
            window=16, min_periods=1
        ).std()
    
    def calculate_slope(series):
        if len(series) < 2:
            return np.nan
        x = np.arange(len(series))
        mask = ~np.isnan(series)
        if mask.sum() < 2:
            return np.nan
        slope, _ = np.polyfit(x[mask], series[mask], 1)
        return slope
    
    if 'sbp' in df_features.columns:
        df_features['bp_trend'] = df_features['sbp'].rolling(
            window=12, min_periods=2
        ).apply(calculate_slope, raw=False)
    
    return df_features

## Section 8: Fuse Static Clinical Domain Flags

In [54]:
def load_clinical_domain_flags(senior_id, conn):
    """Load the 11 clinical domain flags from the senior_risk_profiles table."""
    
    query = """
    SELECT * FROM senior_risk_profiles WHERE senior_id = ?
    """
    
    df_flags = pd.read_sql_query(query, conn, params=(str(senior_id),))
    
    if len(df_flags) == 0:
        return None
    
    flag_cols = [col for col in df_flags.columns 
                 if col != 'senior_id' and df_flags[col].dtype in ['int64', 'float64', 'bool']]
    
    return df_flags[['senior_id'] + flag_cols].iloc[0]

def fuse_clinical_flags(df_features, clinical_flags):
    """Join clinical domain flags to each time bucket."""
    
    if df_features is None or clinical_flags is None:
        return df_features
    
    df_fused = df_features.copy()
    
    for col in clinical_flags.index:
        if col != 'senior_id':
            df_fused[col] = clinical_flags[col]
    
    return df_fused

## Section 9: Create Alert Context Feature (Recent Event Burden)

In [55]:
def add_alert_context_feature(df_features, senior_id, conn):
    """Create recent_event_burden features for each time bucket."""
    
    if df_features is None or len(df_features) == 0:
        return df_features
    
    df_context = df_features.copy()
    
    query = """
    SELECT alert_date FROM alerts 
    WHERE senior_id = ? AND severity IN (1, 2, 3)
    ORDER BY alert_date
    """
    alerts_df = pd.read_sql_query(query, conn, params=(str(senior_id),))

    df_features['recent_event_burden'] = 0.0
    
    if len(alerts_df) == 0:
        return df_context
    
    alerts_df['alert_date'] = pd.to_datetime(alerts_df['alert_date'])

    alert_series = pd.Series(0, index=df_features['timestamp'])
    alert_times = alerts_df['alert_date'].dt.round('15min')
    alert_counts = alert_times.value_counts()

    common_indices = alert_series.index.intersection(alert_counts.index)
    alert_series.loc[common_indices] = alert_counts.loc[common_indices]

    rolling_burden = alert_series.rolling(window='48h', closed='left').sum()
    
    df_features['recent_event_burden'] = rolling_burden.values
    
    df_features['recent_event_burden'] = df_features['recent_event_burden'].fillna(0)
    
    return df_context

## Section 10: Label Target Variables (Multi-Severity Alert Windows)

In [56]:
def create_target_labels(df_features, senior_id, conn):
    """Create binary target labels for all severity levels based on alerts in the prior 24 hours."""
    
    if df_features is None or len(df_features) == 0:
        return df_features
    
    df_labeled = df_features.copy()
    df_labeled['label_1'] = 0
    df_labeled['label_2'] = 0
    df_labeled['label_3'] = 0
    
    for severity in [1, 2, 3]:
        query = """
        SELECT alert_date FROM alerts 
        WHERE senior_id = ? AND severity = ?
        """
        
        alerts_df = pd.read_sql_query(query, conn, params=(str(senior_id), severity))
        
        if len(alerts_df) == 0:
            continue
        
        alerts_df['alert_date'] = pd.to_datetime(alerts_df['alert_date'])
        alert_dates = alerts_df['alert_date'].values
        
        label_col = f'label_{severity}'
        for alert_ts in alert_dates:
            window_start = pd.Timestamp(alert_ts) - timedelta(hours=24)
            window_end = pd.Timestamp(alert_ts)
            
            mask = (df_labeled['timestamp'] >= window_start) & \
                   (df_labeled['timestamp'] < window_end)
            df_labeled.loc[mask, label_col] = 1
    
    return df_labeled

## Section 11: Process Senior-by-Senior and Save to Parquet

In [59]:
def process_senior_complete_pipeline(senior_id, conn):
    """Complete pipeline for a single senior"""
    try:
        df_resampled = resample_senior_vitals(senior_id, conn)
        if df_resampled is None or len(df_resampled) == 0:
            return None

        df_wide = pivot_to_wide_format(df_resampled)
        if df_wide is None:
            return None

        df_filled = apply_forward_fill_imputation(df_wide)

        df_signals = engineer_signal_features(df_filled)

        clinical_flags = load_clinical_domain_flags(senior_id, conn)
        if clinical_flags is not None:
            df_signals = fuse_clinical_flags(df_signals, clinical_flags)

        df_context = add_alert_context_feature(df_signals, senior_id, conn)

        df_final = create_target_labels(df_context, senior_id, conn)

        return df_final

    except Exception as e:
        print(f"Error processing senior {senior_id}: {str(e)}")
        import traceback
        traceback.print_exc()
        return None

In [64]:
batch_size = 500
current_batch = []
successful_seniors = 0
failed_seniors = 0

temp_dir = '../data/processed/temp_chunks'
os.makedirs(temp_dir, exist_ok=True)
output_path = '../data/processed/multimodal_features.parquet'

start_time = time.time()

print(f"Starting processing for {len(gold_seniors_list)} seniors...")
print(f"Intermediate chunks will be saved to: {temp_dir}")

for idx, senior_id in enumerate(gold_seniors_list):
    senior_start = time.time()
    
    try:
        df_senior = process_senior_complete_pipeline(senior_id, conn)
        
        if df_senior is not None and len(df_senior) > 0:
            current_batch.append(df_senior)
            successful_seniors += 1
        else:
            failed_seniors += 1
            if idx < 10: 
                print(f" - Senior {senior_id} returned no data")
            
    except Exception as e:
        failed_seniors += 1
        print(f"  Error processing senior {senior_id}: {e}")

    if (idx + 1) % 50 == 0:
        elapsed = (time.time() - start_time) / 60
        print(f"Progress: {idx + 1}/{len(gold_seniors_list)} | Success: {successful_seniors} | Failed: {failed_seniors} | Time: {elapsed:.1f}m")

    if len(current_batch) >= batch_size:
        batch_df = pd.concat(current_batch, ignore_index=True)
        chunk_path = f"{temp_dir}/chunk_{idx}.parquet"
        batch_df.to_parquet(chunk_path, index=False)
        print(f"  Saved batch to {chunk_path} (Rows: {len(batch_df)})")
        current_batch = []

if current_batch:
    batch_df = pd.concat(current_batch, ignore_index=True)
    batch_df.to_parquet(f"{temp_dir}/chunk_final.parquet", index=False)
    print(f"  Saved final batch (Rows: {len(batch_df)})")

print("\nMerging all chunks into final file...")
all_chunks = glob.glob(f"{temp_dir}/*.parquet")

if all_chunks:
    full_df = pd.concat([pd.read_parquet(f) for f in all_chunks], ignore_index=True)
    full_df.to_parquet(output_path, index=False)
    
    print("\n Process Complete!")
    print(f"Final Matrix Shape: {full_df.shape}")
    print(f"Output saved to: {output_path}")
    
    for f in all_chunks:
        os.remove(f)
    os.rmdir(temp_dir)
else:
    print("No data was generated.")

print(f"Total Success: {successful_seniors:,} | Total Failed: {failed_seniors:,}")

Starting processing for 8268 seniors...
Intermediate chunks will be saved to: ../data/processed/temp_chunks
Progress: 50/8268 | Success: 50 | Failed: 0 | Time: 0.8m
Progress: 100/8268 | Success: 100 | Failed: 0 | Time: 1.9m
Progress: 150/8268 | Success: 150 | Failed: 0 | Time: 3.0m
Progress: 200/8268 | Success: 200 | Failed: 0 | Time: 4.0m
Progress: 250/8268 | Success: 250 | Failed: 0 | Time: 5.2m
Progress: 300/8268 | Success: 300 | Failed: 0 | Time: 6.3m
Progress: 350/8268 | Success: 350 | Failed: 0 | Time: 7.5m
Progress: 400/8268 | Success: 400 | Failed: 0 | Time: 8.3m
Progress: 450/8268 | Success: 450 | Failed: 0 | Time: 9.4m
Progress: 500/8268 | Success: 500 | Failed: 0 | Time: 10.5m
  Saved batch to ../data/processed/temp_chunks/chunk_499.parquet (Rows: 1269788)
Progress: 550/8268 | Success: 550 | Failed: 0 | Time: 11.6m
Progress: 600/8268 | Success: 600 | Failed: 0 | Time: 12.2m
Progress: 650/8268 | Success: 650 | Failed: 0 | Time: 12.7m
Progress: 700/8268 | Success: 700 | Failed

## Section 12: Feature Matrix Exploration and Statistics

In [65]:
df_matrix = pd.read_parquet(output_path)

print("=" * 100)
print("FEATURE MATRIX DETAILED EXPLORATION")
print("=" * 100)

print(f"\nShape: {df_matrix.shape}")
print(f"Unique Seniors: {df_matrix['senior_id'].nunique():,}")
print(f"Date Range: {df_matrix['timestamp'].min()} to {df_matrix['timestamp'].max()}")

print("\nData Type Summary:")
print(df_matrix.dtypes.value_counts())

print("\nMissing Values Summary:")
missing = df_matrix.isnull().sum()
if missing.sum() > 0:
    print(missing[missing > 0].sort_values(ascending=False))
else:
    print("No missing values!")

print("\nVital Signs Statistics:")
vital_cols = [col for col in df_matrix.columns 
              if col in ['heartrate', 'temperature', 'saturation', 'steps', 'sbp', 'dbp']]
if len(vital_cols) > 0:
    print(df_matrix[vital_cols].describe())

print("\nEngineered Features Statistics:")
engineered_cols = [col for col in df_matrix.columns 
                   if col in ['hr_volatility', 'bp_trend', 'pulse_pressure', 'recent_event_burden']]
if len(engineered_cols) > 0:
    print(df_matrix[engineered_cols].describe())

print("\nTarget Label Statistics:")
for label_col in ['label_1', 'label_2', 'label_3']:
    pos_count = (df_matrix[label_col] == 1).sum()
    neg_count = (df_matrix[label_col] == 0).sum()
    pos_pct = 100 * pos_count / len(df_matrix)
    print(f"\n  {label_col}:")
    print(f"    Total samples: {len(df_matrix):,}")
    print(f"    Positive cases ({label_col}=1): {pos_count:,} ({pos_pct:.2f}%)")
    print(f"    Negative cases ({label_col}=0): {neg_count:,} ({100*neg_count/len(df_matrix):.2f}%)")
    if pos_count > 0:
        print(f"    Class balance ratio: 1:{neg_count / pos_count:.1f}")

print("\n" + "=" * 100)
print("FEATURE ENGINEERING COMPLETE")
print("=" * 100)
print(f"✓ Output file: {output_path}")
print(f"✓ Total samples: {len(df_matrix):,}")
print(f"✓ Total features: {len(df_matrix.columns)}")

# conn.close()
# print("\n✓ Database connection closed")

FEATURE MATRIX DETAILED EXPLORATION

Shape: (20783924, 25)
Unique Seniors: 8,268
Date Range: 2025-11-01 00:00:00 to 2025-11-30 23:45:00

Data Type Summary:
float64           20
int64              4
datetime64[ns]     1
Name: count, dtype: int64

Missing Values Summary:
saturation               9123081
temperature              8989149
pulse_pressure           8969487
heartrate                8966281
dbp                      8966281
sbp                      8966281
metabolic_endocrine      7611960
cardiovascular           7611960
neurological             7611960
respiratory              7611960
musculoskeletal          7611960
psychiatric_cognitive    7611960
oncological              7611960
sensory                  7611960
other_functional_risk    7611960
gastro_renal_urologic    7611960
other                    7611960
bp_trend                 6971453
hr_volatility            6501705
steps                    1314515
dtype: int64

Vital Signs Statistics:
        temperature     heartrat